# KrishiDisha assistant - QLoRA fine-tuning on Kaggle

Trains the two candidate bases (Qwen2.5-3B-Instruct, Gemma-3-4B-it) on `data/llm/train.jsonl` with Unsloth + TRL, then merges and exports GGUF for Ollama.

**Setup**: Accelerator GPU T4 x2 (or P100), Internet ON. Upload `data/llm/train.jsonl` + `eval.jsonl` as a private Kaggle dataset and attach it. For Gemma accept the licence on Hugging Face and add `HF_TOKEN` as a Kaggle secret.

One epoch of ~30k examples is roughly 3.5-4.5 h on a T4 for the 3B model; keep `--save-steps` so a session that dies resumes with `--resume`.

In [ ]:
REPO = "https://github.com/shivpratapsinghpanwar/KrishiDisha.ai.git"
BRANCH = "main"
BASE = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"      # or unsloth/gemma-3-4b-it-unsloth-bnb-4bit
MERGE_BASE = "Qwen/Qwen2.5-3B-Instruct"           # unquantized base for the merge (google/gemma-3-4b-it)
EPOCHS = 2

import os, subprocess, sys, glob
if not os.path.exists("/kaggle/working/krishidisha"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, "/kaggle/working/krishidisha"], check=True)
os.chdir("/kaggle/working/krishidisha")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth", "trl", "peft", "bitsandbytes", "datasets", "pyyaml"], check=True)
train = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)[0]
evalf = glob.glob("/kaggle/input/**/eval.jsonl", recursive=True)[0]
print(train, evalf)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("no HF_TOKEN secret (fine for Qwen):", e)

In [ ]:
# smoke test first (2 minutes): 200 examples, 0.05 epoch
!python -m ml.llm.train_llm --data {train} --eval {evalf} --base {BASE} --out /kaggle/working/kd-smoke --epochs 0.05 --limit 200

In [ ]:
OUT = "/kaggle/working/kd-" + ("gemma" if "gemma" in BASE.lower() else "qwen")
!python -m ml.llm.train_llm --data {train} --eval {evalf} --base {BASE} --out {OUT} --epochs {EPOCHS} --save-steps 200 --resume
print(open(f"{OUT}/sample_generations.md").read()[:4000])

In [ ]:
# Merge + GGUF (needs llama.cpp; ~10 min). Download the GGUF and Modelfile from the Output tab, then locally:
#   ollama create krishidisha -f models/llm/Modelfile.qwen
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /kaggle/working/llama.cpp && pip install -q -r /kaggle/working/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!cd /kaggle/working/llama.cpp && cmake -B build -DGGML_CUDA=OFF > /dev/null && cmake --build build --target llama-quantize -j > /dev/null
!python -m ml.llm.export_gguf --adapter {OUT}/adapter --base {MERGE_BASE} --out /kaggle/working/gguf --quant q4_k_m q8_0 --llama-cpp /kaggle/working/llama.cpp
!ls -la /kaggle/working/gguf